# JPMorgan London Whale Risk Simulation

This notebook simulates a simplified credit derivatives portfolio inspired by the JPMorgan London Whale incident.

We will:
- Generate synthetic daily returns for a CDS portfolio.
- Compute **Value-at-Risk (VaR)** using Parametric, Historical, and Monte Carlo methods.
- Compute **Expected Shortfall (CVaR)** for 95% and 99% confidence levels.
- Visualize return distributions and stress scenarios.
- Discuss why VaR failed and why CVaR matters.


## Historical Context: The JPMorgan London Whale Incident

The 'London Whale' incident refers to a significant trading loss incurred by JPMorgan Chase in 2012. A trader in the bank's London office, Bruno Iksil (nicknamed the 'London Whale' due to the size of his trades), made massive bets on credit default swaps (CDS). These positions were so large they moved the market, and when the market turned against them, the losses mounted rapidly, eventually totaling over $6 billion.

Key issues and lessons learned from this event include:

*   **Flawed Risk Models:** JPMorgan Chase's Value-at-Risk (VaR) models, like those we're simulating here, severely underestimated the potential losses. The models did not adequately capture the tail risk or the illiquidity of the markets for such large positions.
*   **Concentration Risk:** The sheer size of Iksil's positions meant that a single trader could have a disproportionate impact on the bank's overall risk profile.
*   **Lack of Oversight:** There was a failure in the bank's internal controls and oversight mechanisms to detect and curb the excessive risk-taking in a timely manner.
*   **Regulatory Scrutiny:** The incident led to increased scrutiny from regulators globally, contributing to stricter regulations under Basel III and the Dodd-Frank Act, particularly regarding capital requirements and stress testing for banks.

In essence, the London Whale incident highlighted how even sophisticated financial institutions could suffer massive losses due to inadequacies in their risk management frameworks, especially their reliance on metrics like VaR which can fail to predict extreme tail events. It underscored the critical importance of robust risk management and the use of more comprehensive metrics like Expected Shortfall (CVaR) to capture extreme potential losses.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Simulation parameters
np.random.seed(42)
days = 1000  # number of simulated days
portfolio_mean = 0.0002  # daily mean return (~5% annualized)
portfolio_vol = 0.02     # daily volatility (~32% annualized)

# Simulate normal market returns
returns = np.random.normal(portfolio_mean, portfolio_vol, days)

# Add stress scenario: 5% of days have extreme losses
stress_days = int(0.05 * days)
stress_losses = np.random.normal(-0.08, 0.03, stress_days)
returns[:stress_days] = stress_losses

# Convert to DataFrame
df = pd.DataFrame({'Returns': returns})


## Q1 — VaR vs Expected Shortfall

We compare different risk measures:

- Parametric VaR  
- Historical VaR  
- Monte Carlo VaR  
- Expected Shortfall (ES)

These measures attempt to quantify portfolio risk under normal conditions.

In [2]:
# Confidence levels
conf_levels = [0.95, 0.99]

# Parametric VaR (Variance-Covariance)
parametric_var = {cl: portfolio_mean + portfolio_vol * np.percentile(np.random.normal(0,1,100000), (1-cl)*100) for cl in conf_levels}

# Historical VaR
historical_var = {cl: np.percentile(df['Returns'], (1-cl)*100) for cl in conf_levels}

# Monte Carlo VaR
mc_sim = np.random.normal(portfolio_mean, portfolio_vol, 100000)
monte_carlo_var = {cl: np.percentile(mc_sim, (1-cl)*100) for cl in conf_levels}

# Expected Shortfall (CVaR)
historical_es = {cl: df[df['Returns'] <= historical_var[cl]]['Returns'].mean() for cl in conf_levels}
monte_carlo_es = {cl: mc_sim[mc_sim <= monte_carlo_var[cl]].mean() for cl in conf_levels}

# Display results
print("VaR and Expected Shortfall Estimates:\n")
for cl in conf_levels:
    print(f"Confidence Level: {int(cl*100)}%")
    print(f" Parametric VaR: {parametric_var[cl]:.4f}")
    print(f" Historical VaR: {historical_var[cl]:.4f}")
    print(f" Monte Carlo VaR: {monte_carlo_var[cl]:.4f}")
    print(f" Historical ES: {historical_es[cl]:.4f}")
    print(f" Monte Carlo ES: {monte_carlo_es[cl]:.4f}\n")

VaR and Expected Shortfall Estimates:

Confidence Level: 95%
 Parametric VaR: -0.0327
 Historical VaR: -0.0415
 Monte Carlo VaR: -0.0327
 Historical ES: -0.0740
 Monte Carlo ES: -0.0412

Confidence Level: 99%
 Parametric VaR: -0.0461
 Historical VaR: -0.0945
 Monte Carlo VaR: -0.0465
 Historical ES: -0.1102
 Monte Carlo ES: -0.0537



### ✅ Q1 Response

1. How does VaR differ from Expected Shortfall?  
2. Which measure better captures extreme losses?  
3. Why might VaR underestimate tail risk in this simulation?

## Q2 — Risk Under Stress vs Normal Conditions

We compare risk metrics:

- Before stress (normal market conditions)
- During stress (tail-loss scenario)

This reveals how models behave under changing environments.

## VaR and ES Calculation and Plot Prior to Stress Event

To understand the impact of the stress scenario, let's analyze the VaR and ES metrics using only the market returns *before* the extreme losses were introduced. This provides a baseline risk assessment.

In [3]:
# Isolate returns prior to stress days
pre_stress_returns = returns[stress_days:]
df_pre_stress = pd.DataFrame({'Returns': pre_stress_returns})

# Recalculate mean and volatility for the pre-stress period
pre_stress_mean = df_pre_stress['Returns'].mean()
pre_stress_vol = df_pre_stress['Returns'].std()

print(f"Pre-stress Mean Return: {pre_stress_mean:.4f}")
print(f"Pre-stress Volatility: {pre_stress_vol:.4f}")

# Parametric VaR (Variance-Covariance) for pre-stress
parametric_var_pre_stress = {cl: pre_stress_mean + pre_stress_vol * np.percentile(np.random.normal(0,1,100000), (1-cl)*100) for cl in conf_levels}

# Historical VaR for pre-stress
historical_var_pre_stress = {cl: np.percentile(df_pre_stress['Returns'], (1-cl)*100) for cl in conf_levels}

# Monte Carlo VaR for pre-stress (simulate using pre-stress mean/vol)
mc_sim_pre_stress = np.random.normal(pre_stress_mean, pre_stress_vol, 100000)
monte_carlo_var_pre_stress = {cl: np.percentile(mc_sim_pre_stress, (1-cl)*100) for cl in conf_levels}

# Expected Shortfall (CVaR) for pre-stress
historical_es_pre_stress = {cl: df_pre_stress[df_pre_stress['Returns'] <= historical_var_pre_stress[cl]]['Returns'].mean() for cl in conf_levels}
monte_carlo_es_pre_stress = {cl: mc_sim_pre_stress[mc_sim_pre_stress <= monte_carlo_var_pre_stress[cl]].mean() for cl in conf_levels}

# Display results
print("\nVaR and Expected Shortfall Estimates (Pre-Stress):\n")
for cl in conf_levels:
    print(f"Confidence Level: {int(cl*100)}%")
    print(f" Parametric VaR: {parametric_var_pre_stress[cl]:.4f}")
    print(f" Historical VaR: {historical_var_pre_stress[cl]:.4f}")
    print(f" Monte Carlo VaR: {monte_carlo_var_pre_stress[cl]:.4f}")
    print(f" Historical ES: {historical_es_pre_stress[cl]:.4f}")
    print(f" Monte Carlo ES: {monte_carlo_es_pre_stress[cl]:.4f}\n")

Pre-stress Mean Return: 0.0008
Pre-stress Volatility: 0.0196

VaR and Expected Shortfall Estimates (Pre-Stress):

Confidence Level: 95%
 Parametric VaR: -0.0314
 Historical VaR: -0.0302
 Monte Carlo VaR: -0.0315
 Historical ES: -0.0381
 Monte Carlo ES: -0.0396

Confidence Level: 99%
 Parametric VaR: -0.0447
 Historical VaR: -0.0419
 Monte Carlo VaR: -0.0447
 Historical ES: -0.0497
 Monte Carlo ES: -0.0515



### ✅ Q2 Response

1. How do VaR and ES change under stress?  
2. Which metric increases more significantly?  
3. What does this suggest about model sensitivity?

## Q3 — Model Risk and Failure

The London Whale losses highlight weaknesses in:

- Overreliance on VaR  
- Model assumptions  
- Tail risk estimation  

In [6]:
# Visualization: Pre-Stress Return distribution with VaR & ES markers
fig_pre_stress = px.histogram(df_pre_stress, x='Returns', nbins=50, title='Portfolio Return Distribution (Pre-Stress) with VaR & ES')

# Calculate histogram data manually to get max_y reliably for pre-stress plot
hist_counts_pre_stress, hist_bins_pre_stress = np.histogram(df_pre_stress['Returns'], bins=50)
max_y_reliable_pre_stress = hist_counts_pre_stress.max() if hist_counts_pre_stress.size > 0 else 1

for i, cl in enumerate(conf_levels):
    # Add VaR line as a Scatter trace
    fig_pre_stress.add_trace(go.Scatter(
        x=[historical_var_pre_stress[cl], historical_var_pre_stress[cl]],
        y=[0, max_y_reliable_pre_stress],
        mode='lines',
        line=dict(color=colors[i], width=2),
        name=f"VaR {int(cl*100)}% (Pre-Stress)",
        showlegend=True
    ))
    # Add ES line as a Scatter trace
    fig_pre_stress.add_trace(go.Scatter(
        x=[historical_es_pre_stress[cl], historical_es_pre_stress[cl]],
        y=[0, max_y_reliable_pre_stress],
        mode='lines',
        line=dict(color=colors[i], width=2, dash='dash'),
        name=f"ES {int(cl*100)}% (Pre-Stress)",
        showlegend=True
    ))

# Ensure legend is shown for this plot and remove grey background
fig_pre_stress.update_layout(plot_bgcolor='white', showlegend=True)
fig_pre_stress.show()


In [7]:
# Visualization: Return distribution with VaR & ES markers
fig = px.histogram(df, x='Returns', nbins=50, title='Portfolio Return Distribution with VaR & ES')
colors = ['red', 'orange']

# Calculate histogram data manually to get max_y reliably
hist_counts, hist_bins = np.histogram(df['Returns'], bins=50)
max_y_reliable = hist_counts.max() if hist_counts.size > 0 else 1 # Fallback if no counts

for i, cl in enumerate(conf_levels):
    # Add VaR line as a Scatter trace
    fig.add_trace(go.Scatter(
        x=[historical_var[cl], historical_var[cl]],
        y=[0, max_y_reliable],
        mode='lines',
        line=dict(color=colors[i], width=2),
        name=f"VaR {int(cl*100)}%",
        showlegend=True
    ))
    # Add ES line as a Scatter trace
    fig.add_trace(go.Scatter(
        x=[historical_es[cl], historical_es[cl]],
        y=[0, max_y_reliable],
        mode='lines',
        line=dict(color=colors[i], width=2, dash='dash'),
        name=f"ES {int(cl*100)}%",
        showlegend=True
    ))

# Ensure legend is shown for this plot and remove grey background
fig.update_layout(plot_bgcolor='white', showlegend=True)
fig.show()

# Stress scenario impact visualization
stress_fig = go.Figure()
stress_fig.add_trace(go.Scatter(y=returns, mode='lines', name='Daily Returns'))
stress_fig.update_layout(
    title='Simulated Returns with Stress Scenario',
    xaxis_title='Day',
    yaxis_title='Return',
    plot_bgcolor='white',
    showlegend=True
)
stress_fig.show()

### ✅ Q3 Response

1. Why did VaR fail to capture the true risk exposure?  
2. How did position size and liquidity affect outcomes?  
3. How does this simulation mirror real-world failure?

## Q4 — Risk Management and Governance


The London Whale incident reflects failures beyond the model:

- Oversized positions  
- Weak oversight  
- Poor communication  
- Inadequate controls  

### ✅ Q4 Response

Provide at least THREE recommendations:

- One related to risk modeling  
- One related to controls or limits  
- One related to governance or oversight  

Explain why each is necessary.

## Final Interpretation — London Whale

Why was reliance on VaR dangerous?

Use:
- Q1 results  
- Q2 comparisons  
- Q3 visualizations  